In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Inputs and CONSTANTS
angle = np.array([...]) 
I1 = np.array([...]) 
I2 = np.array([...]) 
I3 = np.array([...]) 
I4 = np.array([...]) 
I5 = np.array([...]) 

delta_theta_deg = 1 # Uncertainty in angle measurement (degrees)

I = np.vstack([I1, I2, I3, I4, I5]) 

I_mean = np.mean(I, axis=0) 
I_std = np.std(I, axis=0, ddof=1) 
I_err = I_std / np.sqrt(I.shape[0]) 

# 4th Degree Quad Fit
coeff = np.polyfit(angle, I_mean, 4, w=1/I_err) 
p = np.poly1d(coeff) 
xfit = np.linspace(min(angle), max(angle), 500) 
yfit = p(xfit) 

# Take the derivative of the polynomial: dp/dx
dp = p.deriv()

# Find the roots of the derivative (where slope = 0)
roots = dp.roots

# Keep only the real roots (ignore imaginary/complex roots)
real_roots = roots[np.isreal(roots)].real

# Keep only the roots that fall within our measured angle range
valid_roots = real_roots[(real_roots >= min(angle)) & (real_roots <= max(angle))]

# The Brewster angle is the root that gives the lowest intensity
theta_B = valid_roots[np.argmin(p(valid_roots))]

# Calculate Refractive Index
mu = np.tan(np.deg2rad(theta_B))

# Error Propagation
delta_theta_rad = np.deg2rad(delta_theta_deg)
mu_error = (1 + mu**2) * delta_theta_rad

# Plotting
plt.figure(figsize=(10, 6))
plt.errorbar(angle, I_mean, yerr=I_err, fmt='o', color='blue', label='Data ± SEM')
plt.plot(xfit, yfit, color='black', label='Fit', linestyle='--')

# Plot a vertical line at the exact minimum we calculated
plt.axvline(theta_B, color='red', linestyle=':', label=f'Minimum at {theta_B:.2f}°')

plt.xlabel('Incident Angle (degrees)')
plt.ylabel('Current (microamps)')
plt.title('Current vs Incident Angle (4th Degree Fit)')
plt.legend()
plt.grid()
plt.show()

print(f"Brewster Angle (theta_B): {theta_B:.3f} degrees")
print(f"Refractive Index (mu):    {mu:.3f} ± {mu_error:.3f}")
print(f"Standard Deviation of Intensities:\n{np.round(I_std, 1)}")
print(f"Standard Error of Intensities:\n{np.round(I_err, 1)}")